## 1. Setup and first look at the raw file

Loading the LendingClub accepted-loans file. The first row count fails with a `UnicodeDecodeError` - the file contains bytes the default Windows codec cannot read - so the next cell re-opens it as UTF-8 with errors ignored. The failure is left in deliberately: it is the first thing the raw data does.

In [1]:
print("Hello Umer")

Hello Umer


In [2]:
import os
os.getcwd()

'C:\\Users\\DELL\\Documents'

In [3]:
import os
os.listdir()


['.ipynb_checkpoints',
 'accepted_2007_to_2018Q4.csv',
 'anaconda_projects',
 'credit_risk_training.csv',
 'current_loans_scoring.csv',
 'Custom Office Templates',
 'desktop.ini',
 'My Music',
 'My Pictures',
 'My Videos',
 'Untitled.ipynb']

In [4]:
import pandas as pd

df = pd.read_csv(
    "accepted_2007_to_2018Q4.csv",
    nrows=5,
    low_memory=False
)

df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.shape


(5, 151)

In [6]:
sum(1 for line in open("accepted_2007_to_2018Q4.csv")) - 1


UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 8182: character maps to <undefined>

In [ ]:
sum(1 for line in open("accepted_2007_to_2018Q4.csv", encoding="utf-8", errors="ignore")) - 1

In [ ]:
'loan_status' in df.columns

In [ ]:
df[['loan_status']].head()

## 2. Splitting resolved loans from live ones

Reading a workable sample, then splitting it in two. Loans that have reached a final outcome (Fully Paid or Charged Off) become the training set; loans still running become the scoring set. A loan that has not finished has no answer to learn from.

In [8]:
df_sample = pd.read_csv(
    "accepted_2007_to_2018Q4.csv",
    nrows=100000,
    low_memory=False
)

In [9]:
df_sample.shape

(100000, 151)

In [ ]:
df_sample['loan_status'].value_counts()

In [ ]:

[col for col in df_sample.columns if 'recover' in col.lower()]

In [ ]:
df_sample.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
(df_sample.isnull().sum() / len(df_sample) * 100)\
.sort_values(ascending=False)\
.head(20)

In [ ]:
df_sample.info()

In [ ]:
df_sample.info(show_counts=True)

In [ ]:
df_sample[['loan_amnt','int_rate','grade','sub_grade','annual_inc','dti','loan_status']].head()

In [ ]:
df_sample.columns.tolist()

In [ ]:
df_sample[['loan_status']].value_counts()

In [ ]:
df_sample['loan_status'].unique()

In [10]:
training_df = df_sample[
    df_sample['loan_status'].isin(['Fully Paid', 'Charged Off'])
]

In [11]:
training_df.shape

(87891, 151)

In [12]:
current_df = df_sample[
    df_sample['loan_status'] == 'Current'
]

In [13]:
current_df.shape

(11402, 151)

## 3. First pass of column drops

Removing columns that carry no usable signal, and the first of the fields a lender would not have at the point of approval.

In [14]:
[c for c in training_df.columns if c.startswith('sec_app_')]

['sec_app_fico_range_low',
 'sec_app_fico_range_high',
 'sec_app_earliest_cr_line',
 'sec_app_inq_last_6mths',
 'sec_app_mort_acc',
 'sec_app_open_acc',
 'sec_app_revol_util',
 'sec_app_open_act_il',
 'sec_app_num_rev_accts',
 'sec_app_chargeoff_within_12_mths',
 'sec_app_collections_12_mths_ex_med',
 'sec_app_mths_since_last_major_derog']

In [15]:
training_df[
    [
        'sec_app_fico_range_low',
        'sec_app_fico_range_high',
        'sec_app_earliest_cr_line',
        'sec_app_inq_last_6mths',
        'sec_app_mort_acc',
        'sec_app_open_acc',
        'sec_app_revol_util',
        'sec_app_open_act_il',
        'sec_app_num_rev_accts',
        'sec_app_chargeoff_within_12_mths',
        'sec_app_collections_12_mths_ex_med',
        'sec_app_mths_since_last_major_derog'
    ]
].isnull().sum()

sec_app_fico_range_low                 87891
sec_app_fico_range_high                87891
sec_app_earliest_cr_line               87891
sec_app_inq_last_6mths                 87891
sec_app_mort_acc                       87891
sec_app_open_acc                       87891
sec_app_revol_util                     87891
sec_app_open_act_il                    87891
sec_app_num_rev_accts                  87891
sec_app_chargeoff_within_12_mths       87891
sec_app_collections_12_mths_ex_med     87891
sec_app_mths_since_last_major_derog    87891
dtype: int64

In [58]:
drop_cols = [
    'sec_app_fico_range_low',
    'sec_app_fico_range_high',
    'sec_app_earliest_cr_line',
    'sec_app_inq_last_6mths',
    'sec_app_mort_acc',
    'sec_app_open_acc',
    'sec_app_revol_util',
    'sec_app_open_act_il',
    'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog'
]

In [17]:
training_df = training_df.drop(columns=drop_cols)

current_df = current_df.drop(columns=drop_cols)

In [18]:
training_df.shape

(87891, 139)

In [19]:
current_df.shape

(11402, 139)

In [59]:
training_df.to_csv(
    "credit_risk_training.csv",
    index=False
)

In [60]:
current_df.to_csv(
    "current_loans_scoring.csv",
    index=False
)

In [61]:
import os

[f for f in os.listdir() if f.endswith('.csv')]

['accepted_2007_to_2018Q4.csv',
 'credit_risk_training.csv',
 'current_loans_scoring.csv']

In [62]:
[
    col for col in training_df.columns
    if 'pymnt' in col.lower()
]

['pymnt_plan',
 'total_pymnt',
 'total_pymnt_inv',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d']

## 4. Hunting for leakage

Searching the column list for fields that only exist once a loan has resolved - recoveries, payments received, next payment dates, updated credit scores. Anything known only after the fact would let the model predict the past rather than the future.

In [63]:
[col for col in training_df.columns if 'recover' in col.lower()]

['recoveries', 'collection_recovery_fee']

In [64]:
[col for col in training_df.columns if 'rec_' in col.lower()]

['total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'pub_rec_bankruptcies']

In [65]:
[
    col for col in training_df.columns
    if 'fico' in col.lower()
]

['fico_range_low',
 'fico_range_high',
 'last_fico_range_high',
 'last_fico_range_low']

In [66]:
[
    col for col in training_df.columns
    if 'status' in col.lower()
]

['verification_status',
 'loan_status',
 'initial_list_status',
 'verification_status_joint',
 'hardship_status',
 'hardship_loan_status',
 'settlement_status']

In [67]:
[
    col for col in training_df.columns
    if 'settlement' in col.lower()
]

['debt_settlement_flag',
 'debt_settlement_flag_date',
 'settlement_status',
 'settlement_date',
 'settlement_amount',
 'settlement_percentage',
 'settlement_term']

In [68]:
[
    col for col in training_df.columns
    if 'hardship' in col.lower()
]

['hardship_flag',
 'hardship_type',
 'hardship_reason',
 'hardship_status',
 'hardship_amount',
 'hardship_start_date',
 'hardship_end_date',
 'hardship_length',
 'hardship_dpd',
 'hardship_loan_status',
 'hardship_payoff_balance_amount',
 'hardship_last_payment_amount']

In [69]:
[
    col for col in training_df.columns
    if 'date' in col.lower()
]

['hardship_start_date',
 'hardship_end_date',
 'payment_plan_start_date',
 'debt_settlement_flag_date',
 'settlement_date']

In [70]:
[
    col for col in training_df.columns
    if 'last_' in col.lower()
]

['inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'mths_since_last_major_derog',
 'inq_last_12m',
 'hardship_last_payment_amount']

In [71]:
[
    col for col in training_df.columns
    if 'total_' in col.lower()
]

['total_acc',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'total_bal_il',
 'total_rev_hi_lim',
 'total_cu_tl',
 'total_bal_ex_mort',
 'total_bc_limit',
 'total_il_high_credit_limit']

In [72]:
[
    col for col in training_df.columns
    if 'out_' in col.lower()
]

['out_prncp', 'out_prncp_inv']

In [73]:
[col for col in training_df.columns if 'next_' in col.lower()]

['next_pymnt_d']

In [74]:
[col for col in training_df.columns if 'issue' in col.lower()]

['issue_d']

In [75]:
[col for col in training_df.columns if 'term' in col.lower()]

['term', 'deferral_term', 'settlement_term']

## 5. Parsing dates and loan term

Converting `issue_d` from `Mon-YYYY` text into a real date, and extracting the number of months from the `term` field so it can be used as a number.

In [76]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

In [77]:
df['issue_d'].head()
df['issue_d'].dtype

dtype('<M8[ns]')

In [78]:
df['term'].value_counts(dropna=False)

term
60 months    3
36 months    2
Name: count, dtype: int64

In [79]:
df['term_months'] = df['term'].str.extract('(\d+)').astype('Int64')

In [80]:
df[['term', 'term_months']].head()
df['term_months'].value_counts()

term_months
60    3
36    2
Name: count, dtype: Int64

In [81]:
df['term_months'].value_counts(dropna=False)

term_months
60    3
36    2
Name: count, dtype: Int64

In [82]:
df['loan_status'].value_counts(dropna=False)

loan_status
Fully Paid    4
Current       1
Name: count, dtype: int64

In [83]:
df['loan_status'].unique()

array(['Fully Paid', 'Current'], dtype=object)

In [84]:
df.shape

(5, 152)

In [85]:
training_df.shape

(87891, 139)

In [86]:
df_sample.shape

(100000, 151)

In [87]:
current_df.shape

(11402, 151)

In [88]:
current_df = df_sample[
    df_sample['loan_status'] == 'Current'
]

In [89]:
current_df.shape

(11402, 151)

In [90]:
current_df.shape

(11402, 151)

In [91]:
training_df.shape

(87891, 139)

## 6. Joint and secondary-applicant fields

A second drop pass, mostly covering the joint-application and secondary-applicant columns, which are empty for the large majority of loans.

In [95]:
drop_cols = [
    'sec_app_fico_range_low',
    'sec_app_fico_range_high',
    'sec_app_earliest_cr_line',
    'sec_app_inq_last_6mths',
    'sec_app_mort_acc',
    'sec_app_open_acc',
    'sec_app_revol_util',
    'sec_app_open_act_il',
    'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog'
]

In [98]:
training_df = training_df.drop(columns=drop_cols, errors='ignore')
current_df = current_df.drop(columns=drop_cols, errors='ignore')

In [99]:
training_df.shape

(87891, 139)

In [100]:

[col for col in training_df.columns if 'term' in col.lower()]

['term', 'deferral_term', 'settlement_term']

In [101]:
[col for col in training_df.columns if 'flag' in col.lower()]

['hardship_flag', 'debt_settlement_flag', 'debt_settlement_flag_date']

In [102]:
[col for col in training_df.columns if 'joint' in col.lower()]

['annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'revol_bal_joint']

In [103]:
training_df[
    ['annual_inc_joint',
     'dti_joint',
     'verification_status_joint',
     'revol_bal_joint']
].isnull().mean() * 100

annual_inc_joint              99.551717
dti_joint                     99.552855
verification_status_joint     99.551717
revol_bal_joint              100.000000
dtype: float64

In [104]:
[
    col for col in training_df.columns
    if 'joint' in col.lower() or 'verification_status_joint' in col.lower()
]

['annual_inc_joint',
 'dti_joint',
 'verification_status_joint',
 'revol_bal_joint']

In [105]:
training_df[
    ['annual_inc_joint',
     'dti_joint',
     'verification_status_joint',
     'revol_bal_joint']
].describe(include='all')

,annual_inc_joint,dti_joint,verification_status_joint,revol_bal_joint
count,394.000000,393.000000,394,0.0
unique,NaN,NaN,1,NaN
top,NaN,NaN,Not Verified,NaN
freq,NaN,NaN,394,NaN
mean,108787.343071,18.093181,NaN,NaN
std,51197.548851,7.324343,NaN,NaN
min,17950.000000,3.000000,NaN,NaN
25%,76000.000000,13.020000,NaN,NaN
50%,100834.000000,17.600000,NaN,NaN
75%,132580.000000,22.620000,NaN,NaN


In [106]:
training_df[
    ['annual_inc_joint',
     'dti_joint',
     'verification_status_joint',
     'revol_bal_joint']
].isnull().all()

annual_inc_joint             False
dti_joint                    False
verification_status_joint    False
revol_bal_joint               True
dtype: bool

In [107]:
training_df[
    ['annual_inc_joint',
     'dti_joint',
     'verification_status_joint']
].notnull().sum()

annual_inc_joint             394
dti_joint                    393
verification_status_joint    394
dtype: int64

In [108]:
training_df[
    ['annual_inc_joint',
     'dti_joint',
     'verification_status_joint',
     'revol_bal_joint']
].isnull().mean().sort_values(ascending=False) * 100

revol_bal_joint              100.000000
dti_joint                     99.552855
annual_inc_joint              99.551717
verification_status_joint     99.551717
dtype: float64

In [109]:
joint_cols = [
    'annual_inc_joint',
    'dti_joint',
    'verification_status_joint',
    'revol_bal_joint'
]

training_df = training_df.drop(columns=joint_cols)
current_df = current_df.drop(columns=joint_cols)

In [110]:
training_df.shape

(87891, 135)

## 7. Missingness audit

Measuring how empty each column is. Fields that are mostly blank are dropped rather than filled, since imputing a column that is ninety percent missing invents more than it recovers.

In [111]:
missing_pct = training_df.isnull().mean().sort_values(ascending=False) * 100

missing_pct.head(20)

member_id                                     100.000000
next_pymnt_d                                  100.000000
desc                                           99.992036
orig_projected_additional_accrued_interest     99.574473
hardship_status                                99.406083
hardship_dpd                                   99.406083
hardship_start_date                            99.406083
hardship_end_date                              99.406083
payment_plan_start_date                        99.406083
hardship_length                                99.406083
hardship_payoff_balance_amount                 99.406083
hardship_loan_status                           99.406083
deferral_term                                  99.406083
hardship_last_payment_amount                   99.406083
hardship_reason                                99.406083
hardship_type                                  99.406083
hardship_amount                                99.406083
settlement_term                

In [112]:
missing_pct = training_df.isnull().mean().sort_values(ascending=False) * 100

missing_pct[20:40]

settlement_amount              96.677703
settlement_status              96.677703
settlement_percentage          96.677703
mths_since_last_record         81.652274
il_util                        81.272258
mths_since_rcnt_il             79.042223
total_bal_il                   78.444892
open_act_il                    78.444892
open_il_12m                    78.444892
open_il_24m                    78.444892
open_rv_24m                    78.444892
max_bal_bc                     78.444892
all_util                       78.444892
inq_fi                         78.444892
total_cu_tl                    78.444892
inq_last_12m                   78.444892
open_acc_6m                    78.444892
open_rv_12m                    78.444892
mths_since_recent_bc_dlq       74.393283
mths_since_last_major_derog    70.106154
dtype: float64

In [113]:
training_df[
    [
        'settlement_amount',
        'settlement_status',
        'settlement_percentage'
    ]
].notnull().sum()

settlement_amount        2920
settlement_status        2920
settlement_percentage    2920
dtype: int64

In [114]:
training_df[
    [
        'mths_since_last_record',
        'il_util',
        'mths_since_rcnt_il',
        'total_bal_il'
    ]
].notnull().sum()

mths_since_last_record    16126
il_util                   16460
mths_since_rcnt_il        18420
total_bal_il              18945
dtype: int64

In [115]:
training_df[
    [
        'member_id',
        'next_pymnt_d',
        'desc'
    ]
].notnull().sum()

member_id       0
next_pymnt_d    0
desc            7
dtype: int64

In [116]:
drop_cols_2 = [
    'member_id',
    'next_pymnt_d',
    'desc'
]

training_df = training_df.drop(columns=drop_cols_2)
current_df = current_df.drop(columns=drop_cols_2)

In [117]:
training_df.shape

(87891, 132)

In [118]:
training_df[
    [
        'hardship_flag',
        'hardship_status',
        'hardship_amount',
        'hardship_reason'
    ]
].notnull().sum()

hardship_flag      87891
hardship_status      522
hardship_amount      522
hardship_reason      522
dtype: int64

In [119]:
training_df['hardship_flag'].value_counts(dropna=False)

hardship_flag
N    87891
Name: count, dtype: int64

In [120]:
training_df[
    [
        'hardship_status',
        'hardship_amount',
        'hardship_reason'
    ]
].notnull().sum()

hardship_status    522
hardship_amount    522
hardship_reason    522
dtype: int64

In [121]:
training_df['hardship_status'].value_counts(dropna=False)

hardship_status
NaN          87369
COMPLETED      374
BROKEN         148
Name: count, dtype: int64

In [122]:
[col for col in training_df.columns if 'hardship' in col.lower()]

['hardship_flag',
 'hardship_type',
 'hardship_reason',
 'hardship_status',
 'hardship_amount',
 'hardship_start_date',
 'hardship_end_date',
 'hardship_length',
 'hardship_dpd',
 'hardship_loan_status',
 'hardship_payoff_balance_amount',
 'hardship_last_payment_amount']

In [123]:
training_df = training_df.drop(columns=['hardship_flag'])
current_df = current_df.drop(columns=['hardship_flag'])

In [124]:
training_df.shape

(87891, 131)

In [125]:
[col for col in training_df.columns if 'settlement' in col.lower()]

['debt_settlement_flag',
 'debt_settlement_flag_date',
 'settlement_status',
 'settlement_date',
 'settlement_amount',
 'settlement_percentage',
 'settlement_term']

In [126]:
training_df[
    [
        'debt_settlement_flag',
        'debt_settlement_flag_date',
        'settlement_status',
        'settlement_date',
        'settlement_amount',
        'settlement_percentage',
        'settlement_term'
    ]
].notnull().sum()

debt_settlement_flag         87891
debt_settlement_flag_date     2920
settlement_status             2920
settlement_date               2920
settlement_amount             2920
settlement_percentage         2920
settlement_term               2920
dtype: int64

In [127]:
training_df['debt_settlement_flag'].value_counts(dropna=False)

debt_settlement_flag
N    84971
Y     2920
Name: count, dtype: int64

In [128]:
training_df['pymnt_plan'].value_counts(dropna=False)

pymnt_plan
n    87891
Name: count, dtype: int64

In [129]:
training_df = training_df.drop(columns=['pymnt_plan'])
current_df = current_df.drop(columns=['pymnt_plan'])

In [130]:
training_df.shape

(87891, 130)

In [131]:
for col in training_df.columns:
    if training_df[col].nunique(dropna=False) == 1:
        print(col)

out_prncp
out_prncp_inv
policy_code
disbursement_method


In [132]:
drop_cols_3 = [
    'out_prncp',
    'out_prncp_inv',
    'policy_code',
    'disbursement_method'
]

training_df = training_df.drop(columns=drop_cols_3)
current_df = current_df.drop(columns=drop_cols_3)

In [133]:
training_df.shape

(87891, 126)

In [134]:
for col in training_df.columns:
    if training_df[col].nunique(dropna=False) == 1:
        print(col)

## 8. Free-text and identifier columns

Reviewing what is left of the text columns. Free-text fields such as `title` and `emp_title`, and identifiers such as `url` and `zip_code`, are either too high-cardinality to use or carry nothing predictive.

In [135]:
training_df.select_dtypes(include='object').columns.tolist()

['term',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'verification_status',
 'issue_d',
 'loan_status',
 'url',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'earliest_cr_line',
 'initial_list_status',
 'last_pymnt_d',
 'last_credit_pull_d',
 'application_type',
 'hardship_type',
 'hardship_reason',
 'hardship_status',
 'hardship_start_date',
 'hardship_end_date',
 'payment_plan_start_date',
 'hardship_loan_status',
 'debt_settlement_flag',
 'debt_settlement_flag_date',
 'settlement_status',
 'settlement_date']

In [136]:
training_df[
    ['url', 'emp_title', 'title', 'zip_code']
].nunique()

url          87891
emp_title    33659
title           13
zip_code       878
dtype: int64

In [137]:
training_df[['purpose', 'title']].drop_duplicates().head(20)

,purpose,title
0,debt_consolidation,Debt consolidation
1,small_business,Business
2,home_improvement,NaN
4,major_purchase,Major purchase
8,credit_card,Credit card refinancing
12,other,NaN
29,home_improvement,Home improvement
31,debt_consolidation,NaN
37,other,Other
40,house,Home buying


In [138]:
training_df = training_df.drop(columns=['title', 'url'])
current_df = current_df.drop(columns=['title', 'url'])

In [139]:
training_df.shape

(87891, 124)

In [140]:
training_df[['emp_title', 'emp_length']].head(10)

,emp_title,emp_length
0,leadman,10+ years
1,Engineer,10+ years
2,truck driver,10+ years
4,Contract Specialist,3 years
5,Veterinary Tecnician,4 years
6,Vice President of Recruiting Operations,10+ years
7,road driver,10+ years
8,SERVICE MANAGER,6 years
9,Vendor liaison,10+ years
12,Logistics Manager,3 years


In [141]:
training_df['emp_title'].isnull().mean() * 100

np.float64(6.42841701653184)

In [142]:
training_df['zip_code'].isnull().mean() * 100

np.float64(0.0)

In [143]:
training_df['emp_length'].value_counts(dropna=False)

emp_length
10+ years    29041
< 1 year      7782
2 years       7710
3 years       7135
1 year        5787
NaN           5629
5 years       5268
4 years       5062
8 years       4430
6 years       3423
9 years       3357
7 years       3267
Name: count, dtype: int64

In [144]:
training_df['issue_d'].head(10)

0     Dec-2015
1     Dec-2015
2     Dec-2015
4     Dec-2015
5     Dec-2015
6     Dec-2015
7     Dec-2015
8     Dec-2015
9     Dec-2015
12    Dec-2015
Name: issue_d, dtype: object

In [145]:
training_df['issue_d'].nunique()

3

In [146]:
training_df.shape

(87891, 124)

In [147]:
current_df.shape

(11402, 124)

In [148]:
training_df['issue_d'].value_counts()

issue_d
Dec-2015    39002
Nov-2015    32756
Oct-2015    16133
Name: count, dtype: int64

In [149]:
training_df['earliest_cr_line'].head(10)

0     Aug-2003
1     Dec-1999
2     Aug-2000
4     Jun-1998
5     Oct-1987
6     Jun-1990
7     Feb-1999
8     Apr-2002
9     Nov-1994
12    Jun-1996
Name: earliest_cr_line, dtype: object

In [150]:
training_df['earliest_cr_line'].isnull().sum()

np.int64(0)

In [151]:
training_df['last_credit_pull_d'].head(10)

0     Mar-2019
1     Mar-2019
2     Mar-2019
4     Mar-2018
5     May-2017
6     Mar-2019
7     Mar-2019
8     Mar-2019
9     Nov-2018
12    Sep-2018
Name: last_credit_pull_d, dtype: object

In [152]:
training_df['last_credit_pull_d'].isnull().sum()

np.int64(3)

In [153]:
[
    col for col in training_df.columns
    if 'pymnt' in col.lower()
    or 'recover' in col.lower()
    or 'settlement' in col.lower()
]

['total_pymnt',
 'total_pymnt_inv',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'debt_settlement_flag',
 'debt_settlement_flag_date',
 'settlement_status',
 'settlement_date',
 'settlement_amount',
 'settlement_percentage',
 'settlement_term']

In [155]:
training_df = training_df.drop(columns=[
    'total_pymnt',
    'total_pymnt_inv',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term'
])

current_df = current_df.drop(columns=[
    'total_pymnt',
    'total_pymnt_inv',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term'
])

In [157]:
training_df.shape

(87891, 111)

In [158]:
training_df.select_dtypes(include='object').columns.tolist()

['term',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'verification_status',
 'issue_d',
 'loan_status',
 'purpose',
 'zip_code',
 'addr_state',
 'earliest_cr_line',
 'initial_list_status',
 'last_credit_pull_d',
 'application_type',
 'hardship_type',
 'hardship_reason',
 'hardship_status',
 'hardship_start_date',
 'hardship_end_date',
 'payment_plan_start_date',
 'hardship_loan_status']

## 9. Dropping the heavily empty fields

The largest drop pass. Most of these are bureau fields LendingClub only began collecting partway through the period, so their presence says more about when a loan was written than about the borrower.

In [159]:
(training_df.isnull().mean() * 100).sort_values(ascending=False).head(30)

orig_projected_additional_accrued_interest    99.574473
hardship_last_payment_amount                  99.406083
hardship_end_date                             99.406083
hardship_reason                               99.406083
hardship_status                               99.406083
deferral_term                                 99.406083
hardship_amount                               99.406083
hardship_start_date                           99.406083
hardship_type                                 99.406083
payment_plan_start_date                       99.406083
hardship_dpd                                  99.406083
hardship_loan_status                          99.406083
hardship_payoff_balance_amount                99.406083
hardship_length                               99.406083
mths_since_last_record                        81.652274
il_util                                       81.272258
mths_since_rcnt_il                            79.042223
open_il_12m                                   78

In [160]:
training_df = training_df.drop(columns=[
    'orig_projected_additional_accrued_interest',
    'hardship_last_payment_amount',
    'hardship_end_date',
    'hardship_reason',
    'hardship_status',
    'deferral_term',
    'hardship_amount',
    'hardship_start_date',
    'hardship_type',
    'payment_plan_start_date',
    'hardship_dpd',
    'hardship_loan_status',
    'hardship_payoff_balance_amount',
    'hardship_length'
])

current_df = current_df.drop(columns=[
    'orig_projected_additional_accrued_interest',
    'hardship_last_payment_amount',
    'hardship_end_date',
    'hardship_reason',
    'hardship_status',
    'deferral_term',
    'hardship_amount',
    'hardship_start_date',
    'hardship_type',
    'payment_plan_start_date',
    'hardship_dpd',
    'hardship_loan_status',
    'hardship_payoff_balance_amount',
    'hardship_length'
])

In [161]:
training_df.shape

(87891, 97)

In [162]:
(training_df.isnull().mean() * 100).sort_values(ascending=False).head(20)

mths_since_last_record            81.652274
il_util                           81.272258
mths_since_rcnt_il                79.042223
open_il_24m                       78.444892
inq_last_12m                      78.444892
all_util                          78.444892
max_bal_bc                        78.444892
open_rv_24m                       78.444892
open_rv_12m                       78.444892
total_bal_il                      78.444892
open_il_12m                       78.444892
open_act_il                       78.444892
open_acc_6m                       78.444892
total_cu_tl                       78.444892
inq_fi                            78.444892
mths_since_recent_bc_dlq          74.393283
mths_since_last_major_derog       70.106154
mths_since_recent_revol_delinq    63.928047
mths_since_last_delinq            48.050426
mths_since_recent_inq             10.544879
dtype: float64

In [163]:
training_df['loan_status'].value_counts()

loan_status
Fully Paid     70288
Charged Off    17603
Name: count, dtype: int64

In [164]:
training_df.isnull().sum().sum()

np.int64(1296848)

In [165]:
[col for col in training_df.columns if 'date' in col.lower()]

[]

In [166]:
[col for col in training_df.columns if '_d' in col.lower()]

['issue_d',
 'mths_since_last_delinq',
 'last_credit_pull_d',
 'mths_since_last_major_derog',
 'acc_now_delinq',
 'mths_since_recent_bc_dlq',
 'mths_since_recent_revol_delinq',
 'num_tl_90g_dpd_24m',
 'pct_tl_nvr_dlq']

In [167]:
training_df[['loan_status', 'last_credit_pull_d']].head(10)

,loan_status,last_credit_pull_d
0,Fully Paid,Mar-2019
1,Fully Paid,Mar-2019
2,Fully Paid,Mar-2019
4,Fully Paid,Mar-2018
5,Fully Paid,May-2017
6,Fully Paid,Mar-2019
7,Fully Paid,Mar-2019
8,Fully Paid,Mar-2019
9,Fully Paid,Nov-2018
12,Fully Paid,Sep-2018


In [168]:
training_df['last_credit_pull_d'].nunique()

42

In [169]:
training_df = training_df.drop(columns=['last_credit_pull_d'])
current_df = current_df.drop(columns=['last_credit_pull_d'])

In [170]:
training_df.shape

(87891, 96)

In [171]:
training_df.dtypes.value_counts()

float64    80
object     15
int64       1
Name: count, dtype: int64

In [172]:
training_df.select_dtypes(include='object').columns.tolist()

['term',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'verification_status',
 'issue_d',
 'loan_status',
 'purpose',
 'zip_code',
 'addr_state',
 'earliest_cr_line',
 'initial_list_status',
 'application_type']

In [173]:
training_df[['issue_d', 'earliest_cr_line']].head()

,issue_d,earliest_cr_line
0,Dec-2015,Aug-2003
1,Dec-2015,Dec-1999
2,Dec-2015,Aug-2000
4,Dec-2015,Jun-1998
5,Dec-2015,Oct-1987


In [174]:
training_df['issue_d'].value_counts()

issue_d
Dec-2015    39002
Nov-2015    32756
Oct-2015    16133
Name: count, dtype: int64

In [175]:
training_df['earliest_cr_line'].isnull().sum()

np.int64(0)

In [176]:
training_df['earliest_cr_line'].nunique()

619

In [177]:
training_df['issue_d'].nunique()

3

In [178]:
training_df[[
    'term',
    'grade',
    'sub_grade',
    'emp_title',
    'emp_length',
    'home_ownership',
    'verification_status',
    'issue_d',
    'loan_status',
    'purpose',
    'zip_code',
    'addr_state',
    'earliest_cr_line',
    'initial_list_status',
    'application_type'
]].isnull().sum()

term                      0
grade                     0
sub_grade                 0
emp_title              5650
emp_length             5629
home_ownership            0
verification_status       0
issue_d                   0
loan_status               0
purpose                   0
zip_code                  0
addr_state                0
earliest_cr_line          0
initial_list_status       0
application_type          0
dtype: int64

In [179]:
training_df['emp_title'].nunique()

33659

In [180]:
training_df['zip_code'].nunique()

878

In [181]:
 training_df[['emp_title', 'annual_inc']].head(20)

,emp_title,annual_inc
0,leadman,55000.0
1,Engineer,65000.0
2,truck driver,63000.0
4,Contract Specialist,104433.0
5,Veterinary Tecnician,34000.0
6,Vice President of Recruiting Operations,180000.0
7,road driver,85000.0
8,SERVICE MANAGER,85000.0
9,Vendor liaison,42000.0
12,Logistics Manager,64000.0


In [182]:
training_df = training_df.drop(columns=['emp_title'])
current_df = current_df.drop(columns=['emp_title'])

In [183]:
training_df.shape


(87891, 95)

## 10. Feature engineering

Deriving usable numbers from what survived: credit history length in years, employment length as an integer, and issue year and month split out of the issue date.

In [184]:
pd.to_datetime(training_df['issue_d'], format='%b-%Y').head()

0   2015-12-01
1   2015-12-01
2   2015-12-01
4   2015-12-01
5   2015-12-01
Name: issue_d, dtype: datetime64[ns]

In [185]:
pd.to_datetime(training_df['earliest_cr_line'], format='%b-%Y').head()

0   2003-08-01
1   1999-12-01
2   2000-08-01
4   1998-06-01
5   1987-10-01
Name: earliest_cr_line, dtype: datetime64[ns]

In [187]:
issue_dates = pd.to_datetime(training_df['issue_d'], format='%b-%Y')
credit_dates = pd.to_datetime(training_df['earliest_cr_line'], format='%b-%Y')

training_df['credit_history_years'] = (
    (issue_dates - credit_dates).dt.days / 365.25
)

In [188]:
training_df['credit_history_years'].describe()

count    87891.000000
mean        16.730868
std          7.674065
min          3.080082
25%         11.416838
50%         15.164956
75%         20.668036
max         65.913758
Name: credit_history_years, dtype: float64

In [189]:
training_df['emp_length'].value_counts(dropna=False)

emp_length
10+ years    29041
< 1 year      7782
2 years       7710
3 years       7135
1 year        5787
NaN           5629
5 years       5268
4 years       5062
8 years       4430
6 years       3423
9 years       3357
7 years       3267
Name: count, dtype: int64

In [190]:
training_df['emp_length_num'] = (
    training_df['emp_length']
    .str.replace('10+ years', '10', regex=False)
    .str.replace('< 1 year', '0', regex=False)
    .str.extract('(\d+)')[0]
    .astype(float)
    .fillna(-1)
)

In [191]:
training_df['emp_length_num'].value_counts().sort_index()

emp_length_num
-1.0      5629
 0.0      7782
 1.0      5787
 2.0      7710
 3.0      7135
 4.0      5062
 5.0      5268
 6.0      3423
 7.0      3267
 8.0      4430
 9.0      3357
 10.0    29041
Name: count, dtype: int64

In [192]:
training_df['issue_date'] = pd.to_datetime(training_df['issue_d'], format='%b-%Y')

training_df['issue_year'] = training_df['issue_date'].dt.year
training_df['issue_month'] = training_df['issue_date'].dt.month

In [193]:
training_df[['issue_d', 'issue_year', 'issue_month']].head()

,issue_d,issue_year,issue_month
0,Dec-2015,2015,12
1,Dec-2015,2015,12
2,Dec-2015,2015,12
4,Dec-2015,2015,12
5,Dec-2015,2015,12


In [194]:
[
    col for col in training_df.columns
    if 'date' in col.lower()
    or 'year' in col.lower()
]

['credit_history_years', 'issue_date', 'issue_year']

In [195]:
[
    col for col in training_df.columns
    if 'month' in col.lower()
]

['issue_month']

In [196]:
training_df.select_dtypes(include='object').columns.tolist()

['term',
 'grade',
 'sub_grade',
 'emp_length',
 'home_ownership',
 'verification_status',
 'issue_d',
 'loan_status',
 'purpose',
 'zip_code',
 'addr_state',
 'earliest_cr_line',
 'initial_list_status',
 'application_type']

## 11. Applying the same treatment to the live-loan set

Repeating every transform on the scoring set, then reconciling the two column lists so they match exactly. A model cannot score a dataset shaped differently from the one it was trained on.

In [198]:
issue_dates = pd.to_datetime(current_df['issue_d'], format='%b-%Y')
credit_dates = pd.to_datetime(current_df['earliest_cr_line'], format='%b-%Y')

current_df['credit_history_years'] = (
    (issue_dates - credit_dates).dt.days / 365.25
)

In [199]:
current_df['emp_length_num'] = (
    current_df['emp_length']
    .str.replace('10+ years', '10', regex=False)
    .str.replace('< 1 year', '0', regex=False)
    .str.extract('(\d+)')[0]
    .astype(float)
    .fillna(-1)
)

In [200]:
current_df['issue_date'] = pd.to_datetime(
    current_df['issue_d'],
    format='%b-%Y'
)

current_df['issue_year'] = current_df['issue_date'].dt.year
current_df['issue_month'] = current_df['issue_date'].dt.month

In [201]:
set(training_df.columns) - set(current_df.columns)

set()

In [202]:
set(current_df.columns) - set(training_df.columns)

{'earliest_cr_line', 'emp_length', 'issue_d', 'issue_date'}

In [203]:
{'earliest_cr_line', 'emp_length', 'issue_d', 'issue_date'}

{'earliest_cr_line', 'emp_length', 'issue_d', 'issue_date'}

In [204]:
set(current_df.columns) - set(training_df.columns)

{'earliest_cr_line', 'emp_length', 'issue_d', 'issue_date'}

In [206]:
current_df = current_df.drop(columns=[
    'emp_length',
    'earliest_cr_line',
    'issue_d',
    'issue_date'
])

In [207]:
set(current_df.columns) - set(training_df.columns)

set()

In [208]:
set(training_df.columns) - set(current_df.columns)

set()

In [210]:
training_df.shape


(87891, 96)

In [211]:
set(training_df.columns) - set(current_df.columns)

set()

In [212]:
training_df.select_dtypes(include='object').columns.tolist()

['term',
 'grade',
 'sub_grade',
 'home_ownership',
 'verification_status',
 'loan_status',
 'purpose',
 'zip_code',
 'addr_state',
 'initial_list_status',
 'application_type']

## 12. Imputation

Filling what remains, by what the blank actually means: `-1` where missing means the event never happened, `0` where it means none, and the median for genuinely missing numeric readings.

In [213]:
(training_df.isnull().mean() * 100)\
.sort_values(ascending=False)\
.head(20)

mths_since_last_record            81.652274
il_util                           81.272258
mths_since_rcnt_il                79.042223
open_acc_6m                       78.444892
open_act_il                       78.444892
open_il_12m                       78.444892
open_il_24m                       78.444892
total_bal_il                      78.444892
open_rv_24m                       78.444892
open_rv_12m                       78.444892
max_bal_bc                        78.444892
all_util                          78.444892
inq_fi                            78.444892
total_cu_tl                       78.444892
inq_last_12m                      78.444892
mths_since_recent_bc_dlq          74.393283
mths_since_last_major_derog       70.106154
mths_since_recent_revol_delinq    63.928047
mths_since_last_delinq            48.050426
mths_since_recent_inq             10.544879
dtype: float64

In [214]:
training_df[
[
    'mths_since_last_record',
    'il_util',
    'mths_since_rcnt_il',
    'mths_since_last_delinq',
    'mths_since_recent_inq'
]
].isnull().sum()

mths_since_last_record    71765
il_util                   71431
mths_since_rcnt_il        69471
mths_since_last_delinq    42232
mths_since_recent_inq      9268
dtype: int64

In [215]:
training_df[[
    'il_util',
    'open_il_12m',
    'total_bal_il',
    'inq_fi'
]].describe()

,il_util,open_il_12m,total_bal_il,inq_fi
count,16460.000000,18945.000000,18945.000000,18945.000000
mean,71.725334,0.772710,35618.750224,0.948430
std,23.066225,1.007397,42408.787120,1.452089
min,0.000000,0.000000,0.000000,0.000000
25%,59.000000,0.000000,9880.000000,0.000000
50%,75.000000,0.000000,24156.000000,0.000000
75%,88.000000,1.000000,46639.000000,1.000000
max,223.000000,12.000000,878459.000000,17.000000


In [216]:
training_df.isnull().sum().sort_values(ascending=False).head(25)

mths_since_last_record            71765
il_util                           71431
mths_since_rcnt_il                69471
open_acc_6m                       68946
open_act_il                       68946
open_il_12m                       68946
open_il_24m                       68946
total_bal_il                      68946
open_rv_24m                       68946
open_rv_12m                       68946
max_bal_bc                        68946
all_util                          68946
inq_fi                            68946
total_cu_tl                       68946
inq_last_12m                      68946
mths_since_recent_bc_dlq          65385
mths_since_last_major_derog       61617
mths_since_recent_revol_delinq    56187
mths_since_last_delinq            42232
mths_since_recent_inq              9268
num_tl_120dpd_2m                   4590
mo_sin_old_il_acct                 2560
percent_bc_gt_75                    971
bc_util                             944
bc_open_to_buy                      896


In [217]:
minus_one_cols = [
    'mths_since_last_record',
    'mths_since_last_delinq',
    'mths_since_last_major_derog',
    'mths_since_recent_bc_dlq',
    'mths_since_recent_revol_delinq',
    'num_tl_120dpd_2m'
]

training_df[minus_one_cols] = training_df[minus_one_cols].fillna(-1)
current_df[minus_one_cols] = current_df[minus_one_cols].fillna(-1)

In [218]:
zero_cols = [
    'il_util',
    'mths_since_rcnt_il',
    'open_acc_6m',
    'open_act_il',
    'open_il_12m',
    'open_il_24m',
    'total_bal_il',
    'open_rv_24m',
    'open_rv_12m',
    'max_bal_bc',
    'all_util',
    'inq_fi',
    'total_cu_tl',
    'inq_last_12m',
    'mths_since_recent_inq'
]

training_df[zero_cols] = training_df[zero_cols].fillna(0)
current_df[zero_cols] = current_df[zero_cols].fillna(0)

In [219]:
median_cols = [
    'mo_sin_old_il_acct',
    'percent_bc_gt_75',
    'bc_util',
    'bc_open_to_buy'
]

for col in median_cols:
    median_value = training_df[col].median()
    training_df[col] = training_df[col].fillna(median_value)
    current_df[col] = current_df[col].fillna(median_value)

In [220]:
training_df.isnull().sum().sum()

np.int64(897)

In [221]:
current_df.isnull().sum().sum()

np.int64(76)

In [222]:
training_df.isnull().sum().sort_values(ascending=False).head(20)

mths_since_recent_bc              858
revol_util                         36
dti                                 2
num_rev_accts                       1
id                                  0
mo_sin_old_rev_tl_op                0
num_accts_ever_120_pd               0
mths_since_recent_revol_delinq      0
mths_since_recent_inq               0
mths_since_recent_bc_dlq            0
mort_acc                            0
mo_sin_rcnt_tl                      0
mo_sin_rcnt_rev_tl_op               0
mo_sin_old_il_acct                  0
delinq_amnt                         0
chargeoff_within_12_mths            0
bc_util                             0
bc_open_to_buy                      0
avg_cur_bal                         0
acc_open_past_24mths                0
dtype: int64

In [223]:
current_df.isnull().sum().sort_values(ascending=False).head(20)

mths_since_recent_bc              75
revol_util                         1
id                                 0
max_bal_bc                         0
num_accts_ever_120_pd              0
mths_since_recent_revol_delinq     0
mths_since_recent_inq              0
mths_since_recent_bc_dlq           0
mort_acc                           0
mo_sin_rcnt_tl                     0
mo_sin_rcnt_rev_tl_op              0
mo_sin_old_rev_tl_op               0
mo_sin_old_il_acct                 0
delinq_amnt                        0
chargeoff_within_12_mths           0
bc_util                            0
bc_open_to_buy                     0
avg_cur_bal                        0
acc_open_past_24mths               0
inq_last_12m                       0
dtype: int64

In [224]:
training_df['mths_since_recent_bc'] = training_df['mths_since_recent_bc'].fillna(-1)
current_df['mths_since_recent_bc'] = current_df['mths_since_recent_bc'].fillna(-1)

training_df['revol_util'] = training_df['revol_util'].fillna(training_df['revol_util'].median())
current_df['revol_util'] = current_df['revol_util'].fillna(training_df['revol_util'].median())

training_df['dti'] = training_df['dti'].fillna(training_df['dti'].median())
current_df['dti'] = current_df['dti'].fillna(training_df['dti'].median())

training_df['num_rev_accts'] = training_df['num_rev_accts'].fillna(training_df['num_rev_accts'].median())
current_df['num_rev_accts'] = current_df['num_rev_accts'].fillna(training_df['num_rev_accts'].median())

In [225]:
training_df.isnull().sum().sum()

np.int64(0)

In [226]:
current_df.isnull().sum().sum()

np.int64(0)

## 13. Writing the cleaned files

Saving the two cleaned datasets, which become the input to the SQL warehouse and the modelling that follows.

In [227]:
training_df.to_csv(
    'credit_risk_training_clean.csv',
    index=False
)

current_df.to_csv(
    'current_loan_scoring_clean.csv',
    index=False
)